# 폴더 구조
```
prepare_datasets.ipynb - 프로젝트 데이터셋 준비 및 S3 업로드

로컬 폴더 구조:
    ./input/
        ├── config/
        ├── data/
        ├── metadata/
        └── profile/

S3 구조:
    s3://gs-retail-awesome-data-{region}/
        env={env}/
            user={user_id}/
                project={project}/
                    version={version}/
                        ├── config/
                        ├── data/
                        ├── metadata/
                        └── profile/
```

In [1]:
# %% Imports
import os
import boto3
import yaml
import getpass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

In [2]:
# %% Configuration
class DatasetConfig:
    """데이터셋 S3 설정"""
    
    REGION = "us-east-1"
    BUCKET = f"gs-retail-awesome-data-{REGION}"
    ENV = "dev"
    USER_ID = getpass.getuser()  # 현재 시스템 사용자 또는 직접 지정
    PROJECT = "titanic"
    VERSION = "v1.0"
    
    # 로컬 기본 경로
    LOCAL_ROOT = "."
    INPUT_DIR = "input"
    
    # 하위 폴더들
    SUBDIRS = ["config", "data", "metadata", "profile"]
    
    @classmethod
    def get_s3_prefix(
        cls,
        env: str = None,
        user_id: str = None,
        project: str = None,
        version: str = None
    ) -> str:
        """S3 prefix 경로 생성"""
        return (
            f"env={env or cls.ENV}/"
            f"user={user_id or cls.USER_ID}/"
            f"project={project or cls.PROJECT}/"
            f"version={version or cls.VERSION}"
        )
    
    @classmethod
    def get_s3_uri(cls, **kwargs) -> str:
        """전체 S3 URI 반환"""
        prefix = cls.get_s3_prefix(**kwargs)
        return f"s3://{cls.BUCKET}/{prefix}/"
    
    @classmethod
    def get_local_input_path(cls, root: str = None) -> str:
        """로컬 input 경로 반환"""
        return os.path.join(root or cls.LOCAL_ROOT, cls.INPUT_DIR)

In [3]:
# %% Generic Directory & File Functions
def create_directories(base_path: str, subdirs: List[str]) -> Dict[str, str]:
    """
    디렉토리 구조 생성 (generic)
    
    Args:
        base_path: 기본 경로
        subdirs: 생성할 하위 디렉토리 리스트
    
    Returns:
        dict: {subdir_name: full_path}
    """
    paths = {'base': base_path}
    os.makedirs(base_path, exist_ok=True)
    print(f"✓ Created: {base_path}")
    
    for subdir in subdirs:
        full_path = os.path.join(base_path, subdir)
        os.makedirs(full_path, exist_ok=True)
        paths[subdir] = full_path
        print(f"  ✓ Created: {full_path}")
    
    return paths

In [4]:
def create_file(filepath: str, content: any = None, file_type: str = "yaml") -> str:
    """
    파일 생성 (generic)
    
    Args:
        filepath: 파일 전체 경로
        content: 파일 내용 (dict for yaml/json, str for text)
        file_type: 파일 타입 (yaml, json, text, touch)
    
    Returns:
        str: 생성된 파일 경로
    """
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    
    if file_type == "touch" or content is None:
        Path(filepath).touch()
    elif file_type == "yaml":
        with open(filepath, 'w', encoding='utf-8') as f:
            yaml.dump(content, f, default_flow_style=False, allow_unicode=True)
    elif file_type == "json":
        import json
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(content, f, indent=2, ensure_ascii=False)
    else:  # text
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(str(content))
    
    print(f"  ✓ File: {filepath}")
    return filepath


def create_mockup_files(paths: Dict[str, str], mockups: Dict[str, Dict]) -> List[str]:
    """
    목업 파일들 생성 (generic)
    
    Args:
        paths: {subdir_name: full_path} 딕셔너리
        mockups: {subdir_name: {filename: {content, type}}} 딕셔너리
    
    Returns:
        list: 생성된 파일 경로들
    """
    created_files = []
    
    for subdir, files in mockups.items():
        if subdir not in paths:
            print(f"  ⚠ Skipped (path not found): {subdir}")
            continue
        
        print(f"\n[{subdir}]")
        for filename, file_info in files.items():
            filepath = os.path.join(paths[subdir], filename)
            content = file_info.get('content')
            file_type = file_info.get('type', 'yaml')
            create_file(filepath, content, file_type)
            created_files.append(filepath)
    
    return created_files

In [5]:
# %% S3 Functions
def ensure_bucket_exists(bucket, region=None):
    """
    S3 버킷이 없으면 생성 (generic)
    """
    region = region or DatasetConfig.REGION
    s3_client = boto3.client('s3', region_name=region)
    
    try:
        s3_client.head_bucket(Bucket=bucket)
        print(f"✓ Bucket exists: {bucket}")
        return True
    except s3_client.exceptions.ClientError as e:
        error_code = e.response.get('Error', {}).get('Code')
        
        if error_code == '404' or error_code == 'NoSuchBucket':
            print(f"⚠ Bucket not found, creating: {bucket}")
            try:
                if region == 'us-east-1':
                    s3_client.create_bucket(Bucket=bucket)
                else:
                    s3_client.create_bucket(
                        Bucket=bucket,
                        CreateBucketConfiguration={'LocationConstraint': region}
                    )
                print(f"✓ Bucket created: {bucket} (region: {region})")
                return True
            except Exception as create_error:
                print(f"✗ Failed to create bucket: {create_error}")
                return False
        else:
            print(f"✗ Bucket access error: {e}")
            return False

In [6]:
# %% S3 Functions
def upload_to_s3(
    local_path: str,
    bucket: str,
    s3_key: str,
    dry_run: bool = False
) -> str:
    """
    단일 파일 S3 업로드 (generic)
    """
    s3_uri = f"s3://{bucket}/{s3_key}"
    
    if dry_run:
        print(f"  [DRY RUN] {local_path} -> {s3_uri}")
    else:
        s3_client = boto3.client('s3')
        s3_client.upload_file(local_path, bucket, s3_key)
        print(f"  ✓ Uploaded: {s3_uri}")
    
    return s3_uri


def upload_directory_tree(
    local_base: str,
    bucket: str,
    s3_prefix: str,
    dry_run: bool = False
) -> List[str]:
    """
    디렉토리 트리 전체를 S3에 업로드 (generic, 재귀)
    
    Args:
        local_base: 로컬 기본 경로
        bucket: S3 버킷
        s3_prefix: S3 prefix
        dry_run: True면 업로드 시뮬레이션
    
    Returns:
        list: 업로드된 S3 URI들
    """
    uploaded = []
    
    for root, dirs, files in os.walk(local_base):
        for filename in files:
            local_path = os.path.join(root, filename)
            relative_path = os.path.relpath(local_path, local_base)
            s3_key = f"{s3_prefix}/{relative_path}"
            
            uri = upload_to_s3(local_path, bucket, s3_key, dry_run)
            uploaded.append(uri)
    
    return uploaded

In [7]:
# %% Mockup Data Definitions
def get_default_mockups() -> Dict[str, Dict]:
    """
    기본 목업 데이터 정의
    """
    return {
        'config': {
            'config.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'algorithm': {
                        'name': 'lightgbm',
                        'task': 'regression',
                    },
                    'hyperparameters': {
                        'objective': 'regression',
                        'metric': 'rmse',
                        'num_leaves': 31,
                        'learning_rate': 0.05,
                        'n_estimators': 1000,
                    }
                }
            }
        },
        'data': {
            'train.parquet': {'type': 'touch', 'content': None},
            'validation.parquet': {'type': 'touch', 'content': None},
            'test.parquet': {'type': 'touch', 'content': None},
            'input_data_ref.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'data_sources': {
                        'train': {'format': 'parquet', 'rows': 100000},
                        'validation': {'format': 'parquet', 'rows': 10000},
                        'test': {'format': 'parquet', 'rows': 10000},
                    }
                }
            }
        },
        'metadata': {
            'run_manifest.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'created_at': datetime.utcnow().isoformat(),
                    'created_by': DatasetConfig.USER_ID,
                    'project': DatasetConfig.PROJECT,
                    'environment': DatasetConfig.ENV,
                    'execution': {
                        'instance_type': 'ml.m5.xlarge',
                        'instance_count': 1,
                    }
                }
            }
        },
        'profile': {
            'profile.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'profile_name': 'default',
                    'aws': {
                        'region': DatasetConfig.REGION,
                    },
                    'python': {
                        'version': '3.10',
                        'dependencies': [
                            'lightgbm==4.1.0',
                            'pandas==2.0.3',
                            'scikit-learn==1.3.0',
                        ]
                    }
                }
            }
        }
    }

In [8]:
# %% Main Execution Functions
def prepare_local(
    root: str = None,
    subdirs: List[str] = None,
    mockups: Dict[str, Dict] = None
) -> Dict[str, any]:
    """
    로컬 디렉토리 및 목업 파일 생성
    """
    root = root or DatasetConfig.LOCAL_ROOT
    subdirs = subdirs or DatasetConfig.SUBDIRS
    mockups = mockups or get_default_mockups()
    
    input_path = os.path.join(root, DatasetConfig.INPUT_DIR)
    
    print("=" * 60)
    print("Step 1: 로컬 디렉토리 생성")
    print("=" * 60)
    paths = create_directories(input_path, subdirs)
    
    print("\n" + "=" * 60)
    print("Step 2: 목업 파일 생성")
    print("=" * 60)
    files = create_mockup_files(paths, mockups)
    
    return {'paths': paths, 'files': files, 'input_path': input_path}


def upload_to_s3_bucket(local_input_path, env=None, user_id=None, project=None, version=None, dry_run=False):
    s3_prefix = DatasetConfig.get_s3_prefix(env, user_id, project, version)
    s3_uri = DatasetConfig.get_s3_uri(env=env, user_id=user_id, project=project, version=version)
    
    print("\n" + "=" * 60)
    print("Step 3: S3 업로드")
    print("=" * 60)
    print(f"Source: {local_input_path}")
    print(f"Target: {s3_uri}")
    print("-" * 60)
    
    # 버킷 존재 확인 및 생성
    if not dry_run:
        if not ensure_bucket_exists(DatasetConfig.BUCKET, DatasetConfig.REGION):
            raise Exception(f"Failed to ensure bucket exists: {DatasetConfig.BUCKET}")
    
    uploaded = upload_directory_tree(local_input_path, DatasetConfig.BUCKET, s3_prefix, dry_run)
    
    return {'s3_uri': s3_uri, 'uploaded_files': uploaded, 'dry_run': dry_run}


def prepare_and_upload(
    root: str = None,
    user_id: str = None,
    version: str = None,
    dry_run: bool = False,
    mockups: Dict[str, Dict] = None
) -> Dict[str, any]:
    """
    전체 프로세스 실행: 로컬 생성 + S3 업로드
    """
    print("\n" + "=" * 60)
    print("데이터셋 준비 및 S3 업로드")
    print("=" * 60)
    print(f"User: {user_id or DatasetConfig.USER_ID}")
    print(f"Project: {DatasetConfig.PROJECT}")
    print(f"Version: {version or DatasetConfig.VERSION}")
    print(f"Bucket: {DatasetConfig.BUCKET}")
    print("=" * 60)
    
    # Step 1 & 2: 로컬 준비
    local_result = prepare_local(root=root, mockups=mockups)
    
    # Step 3: S3 업로드
    s3_result = upload_to_s3_bucket(
        local_result['input_path'],
        user_id=user_id,
        version=version,
        dry_run=dry_run
    )
    
    print("\n" + "=" * 60)
    print("완료!")
    print(f"S3 Path: {s3_result['s3_uri']}")
    print("=" * 60)
    
    return {
        'local': local_result,
        's3': s3_result
    }

In [9]:
# %% Example Usage

# 방법 1: 전체 프로세스 (dry_run)
result = prepare_and_upload(
    user_id="sean",  # 사용자 ID 지정
    version="v1.0",
    dry_run=False
)


데이터셋 준비 및 S3 업로드
User: sean
Project: titanic
Version: v1.0
Bucket: gs-retail-awesome-data-us-east-1
Step 1: 로컬 디렉토리 생성
✓ Created: ./input
  ✓ Created: ./input/config
  ✓ Created: ./input/data
  ✓ Created: ./input/metadata
  ✓ Created: ./input/profile

Step 2: 목업 파일 생성

[config]
  ✓ File: ./input/config/config.yml

[data]
  ✓ File: ./input/data/train.parquet
  ✓ File: ./input/data/validation.parquet
  ✓ File: ./input/data/test.parquet
  ✓ File: ./input/data/input_data_ref.yml

[metadata]
  ✓ File: ./input/metadata/run_manifest.yml

[profile]
  ✓ File: ./input/profile/profile.yml

Step 3: S3 업로드
Source: ./input
Target: s3://gs-retail-awesome-data-us-east-1/env=dev/user=sean/project=titanic/version=v1.0/
------------------------------------------------------------


/tmp/ipykernel_16407/1836375770.py:47: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat(),


⚠ Bucket not found, creating: gs-retail-awesome-data-us-east-1
✓ Bucket created: gs-retail-awesome-data-us-east-1 (region: us-east-1)
  ✓ Uploaded: s3://gs-retail-awesome-data-us-east-1/env=dev/user=sean/project=titanic/version=v1.0/config/config.yml
  ✓ Uploaded: s3://gs-retail-awesome-data-us-east-1/env=dev/user=sean/project=titanic/version=v1.0/data/input_data_ref.yml
  ✓ Uploaded: s3://gs-retail-awesome-data-us-east-1/env=dev/user=sean/project=titanic/version=v1.0/data/test.parquet
  ✓ Uploaded: s3://gs-retail-awesome-data-us-east-1/env=dev/user=sean/project=titanic/version=v1.0/data/train.parquet
  ✓ Uploaded: s3://gs-retail-awesome-data-us-east-1/env=dev/user=sean/project=titanic/version=v1.0/data/validation.parquet
  ✓ Uploaded: s3://gs-retail-awesome-data-us-east-1/env=dev/user=sean/project=titanic/version=v1.0/metadata/run_manifest.yml
  ✓ Uploaded: s3://gs-retail-awesome-data-us-east-1/env=dev/user=sean/project=titanic/version=v1.0/profile/profile.yml

완료!
S3 Path: s3://gs-re

In [10]:
# 방법 2: 실제 업로드
# result = prepare_and_upload(
#     user_id="sean",
#     version="v1.0",
#     dry_run=False
# )

# 방법 3: 커스텀 목업으로 실행
# custom_mockups = {
#     'config': {
#         'my_config.yml': {
#             'type': 'yaml',
#             'content': {'custom': 'value'}
#         }
#     }
# }
# result = prepare_and_upload(mockups=custom_mockups, dry_run=True)